Here, I verify the correctness of the values of the expectation value and matrix elements, taking into account that the Hamiltonian is being modified, and that the extended swap test is being used.

This is basically a more comprehensive version of `results_measurement1.ipynb`. 

In [1]:
import numpy as np
import pickle
import scipy.sparse
from utils_ferm import (
    op_action_tz
)
from utils_CSF_and_UCSF import (
    orthogonal_transform_obt_tbt,
    obt_phys_spatial_to_spin,
    tbt_phys_spatial_to_spin,
    make_short_H_ferm_op
)
from utils_basic import shift_hamiltonian_qubits
from openfermion import (
    FermionOperator,
    jordan_wigner,
    QubitOperator,
    get_sparse_operator,
    normal_ordered
)

from utils_states import (
    somos_to_seniority_config,
    convert_TZ_format_to_sparse_format,
    variance_of_decomp,
    create_composite_state,
    create_composite_state_prepended,
    expectation,
    matrix_element,
    
)
from utils_hamiltonian_qubit import (
    process_qubit_hamiltonian_to_remove_irrelevant_terms,
    process_qubit_hamiltonian_to_project_onto_symmetric_subspace
)
from utils_hamiltonian_ferm import (
    process_fermionic_hamiltonian_to_remove_irrelevant_terms
)
from utils_partitioning import (
    sorted_insertion_decomposition,
    augment_decomp_with_pauli_x,
    augment_decomp_with_pauli_x_plus_i_pauli_y
)

filename = 'data/h2o_data.dump'

with open(filename,'rb') as f:
    (
    list_CSF,
    list_list_ia_CSF,
    list_list_theta_CSF,
    list_sym_CSF_vec,
    list_UCSF_tz,
    tz_states,
    somos_list,
    psi_GS_UCSF_smik,
    list_orb_rot,
    x_orbrot,
    Enuc,
    obt_spatial,
    tbt_spatial
    ) = pickle.load(f)

#Rotate orbitals 
if len(list_orb_rot) != 0:
    obt, tbt = orthogonal_transform_obt_tbt(x_orbrot,list_orb_rot,obt_spatial,tbt_spatial)
else:
    obt = obt_phys_spatial_to_spin(obt_spatial)
    tbt = tbt_phys_spatial_to_spin(tbt_spatial)


In [2]:
Nqubits = obt.shape[0]
dim     = 2 ** Nqubits
Norb    = Nqubits // 2
Nstates = len(tz_states)

# generate Hamiltonian and obtain its QWC decomposition, for both expectation values and matrix elements

Hferm          = make_short_H_ferm_op(0, obt, tbt)
Hqub           = jordan_wigner(Hferm)
qubit_constant = Hqub.constant
Hqub          -= qubit_constant
Hqub.compress()

qwc_decomp      = sorted_insertion_decomposition(Hqub, 'qwc')
qwc_decomp_swap = augment_decomp_with_pauli_x_plus_i_pauli_y(qwc_decomp, Nqubits)

# generate states and associated state information

configs      = [somos_to_seniority_config(somo, Norb) for somo in somos_list]
statevectors = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in tz_states]

results = {}

for i in range(Nstates):
    for j in range(Nstates):
        if i >= j:

            # load the states 
            bra               = statevectors[i]
            bra_tz            = tz_states[i]
            bra_somos         = somos_list[i]
            bra_config        = configs[i]

            ket               = statevectors[j]
            ket_tz            = tz_states[j]
            ket_somos         = somos_list[j]
            ket_config        = configs[j]

            bra_ket_composite = create_composite_state(bra, ket, Nqubits)

            # process the Hamiltonian for measurements
            Hferm_processed        = process_fermionic_hamiltonian_to_remove_irrelevant_terms(Hferm, Nqubits, bra, ket_tz)
            Hqub_temp              = jordan_wigner(Hferm_processed)
            bra_ket_qubit_constant = Hqub_temp.constant
            Hqub_temp             -= bra_ket_qubit_constant
            Hqub_temp.compress()
            Hqub_processed         = process_qubit_hamiltonian_to_remove_irrelevant_terms(Hqub_temp, Nqubits, bra_config, ket_config)
            Hqub_tapered           = process_qubit_hamiltonian_to_project_onto_symmetric_subspace(Hqub_temp, Nqubits, bra_config, ket_config)

            # obtain Hamiltonian fragments
            qwc_decomp_processed      = sorted_insertion_decomposition(Hqub_processed, 'qwc')
            qwc_decomp_processed_swap = augment_decomp_with_pauli_x_plus_i_pauli_y(qwc_decomp_processed, Nqubits)

            # # do a bunch of correctness checks

            # # first: should get the same matrix element no matter how Hamiltonian is processed
            # matrix_element1 = np.round((bra @ get_sparse_operator(Hferm, Nqubits)                                   @ ket.T)[0,0] , 8)
            # matrix_element2 = np.round((bra @ get_sparse_operator(Hferm_processed, Nqubits)                         @ ket.T)[0,0] , 8)
            # matrix_element3 = np.round((bra @ get_sparse_operator(Hqub + qubit_constant, Nqubits)                   @ ket.T)[0,0] , 8)
            # matrix_element4 = np.round((bra @ get_sparse_operator(Hqub_processed + bra_ket_qubit_constant, Nqubits) @ ket.T)[0,0] , 8)

            # assert matrix_element1 == matrix_element2
            # assert matrix_element1 == matrix_element3
            # assert matrix_element1 == matrix_element4
            # assert matrix_element2 == matrix_element3
            # assert matrix_element2 == matrix_element4
            # assert matrix_element3 == matrix_element4
                
            # # second: should get the same matrix element from the fragments, including with swap test
            # H_from_fragments           = sum(qwc_decomp_processed)
            # H_tensor_1q_from_fragments = sum(qwc_decomp_processed_swap)

            # assert (Hqub_processed - H_from_fragments) == QubitOperator().zero()
            # assert (Hqub_processed * (QubitOperator(f'X{Nqubits}') + 1j * QubitOperator(f'Y{Nqubits}')) 
            #         - H_tensor_1q_from_fragments) == QubitOperator().zero()


            # matrix_element5 = np.round((bra @ get_sparse_operator(H_from_fragments + bra_ket_qubit_constant, Nqubits) @ ket.T)[0,0], 8)
            # if i == j:
            #     cur_op = (
            #         H_tensor_1q_from_fragments 
            #         + bra_ket_qubit_constant
            #     )
            #     matrix_element6 = np.round((bra_ket_composite @ 
            #                                 get_sparse_operator(cur_op, Nqubits + 1) @ 
            #                                 bra_ket_composite.T)[0,0], 8)
            # else:
            #     cur_op = (
            #         H_tensor_1q_from_fragments 
            #         + bra_ket_qubit_constant * QubitOperator(f'X{Nqubits}') 
            #         + 1j * bra_ket_qubit_constant * QubitOperator(f'Y{Nqubits}')
            #     )
            #     matrix_element6 = np.round((bra_ket_composite @ 
            #                                 get_sparse_operator(cur_op, Nqubits + 1) @ 
            #                                 bra_ket_composite.T)[0,0], 8)

            # assert matrix_element1 == matrix_element5
            # assert matrix_element1 == matrix_element6
            # assert matrix_element2 == matrix_element5
            # assert matrix_element2 == matrix_element6
            # assert matrix_element3 == matrix_element5
            # assert matrix_element3 == matrix_element6
            # assert matrix_element4 == matrix_element5
            # assert matrix_element4 == matrix_element6
            # assert matrix_element5 == matrix_element6

            # now calculate the variance of the QWC decomposition for both the unprocessed Hamiltonian and the processed Hamiltonian

            if i == j:
                var_og        = variance_of_decomp(qwc_decomp, ket, Nqubits, general=True)
                var_processed = variance_of_decomp(qwc_decomp_processed, ket, Nqubits, general=True)
            
            else:
                var_og        = variance_of_decomp(qwc_decomp_swap, bra_ket_composite, Nqubits + 1, general=True)
                var_processed = variance_of_decomp(qwc_decomp_processed_swap, bra_ket_composite, Nqubits + 1, general=True)

            results[(i,j)] = (bra_config, ket_config, var_processed, var_og)
            print("{:<2} {:<4} {:<10} {:<15}".format(i, j, np.real(np.round(var_processed, 4)), np.real(np.round(var_og, 4))))


0  0    0.415      36.6772        
1  0    0.9939     4416.2446      
1  1    0.2163     35.3914        
2  0    1.5538     4369.4532      
2  1    1.5195     4086.9281      
2  2    0.2893     40.3724        
3  0    1.6681     4544.7063      
3  1    0.8124     4271.1085      
3  2    0.9704     4226.6055      
3  3    0.8394     44.3712        
4  0    0.5015     4617.7143      
4  1    3.0983     4349.5317      
4  2    0.9805     4304.636       
4  3    0.7355     4488.4388      
4  4    0.4071     39.2445        
5  0    0.818      4336.7895      
5  1    1.9678     4028.6571      
5  2    0.7396     4000.2233      
5  3    1.2194     4185.9551      
5  4    1.1252     4268.7525      
5  5    0.0514     33.1888        
6  0    0.818      4345.7838      
6  1    5.5259     4050.2077      
6  2    0.7499     4009.4285      
6  3    1.199      4194.288       
6  4    1.1271     4276.7435      
6  5    0.5925     3964.8504      
6  6    0.007      34.5521        
7  0    1.0978     4

In [3]:
variance_list_og          = np.array([list(results.values())[i][3] for i in range(len(results.values()))])
variance_og_neyman        = np.sum(variance_list_og ** (1/2)) ** 2

variance_list_processed   = np.array([list(results.values())[i][2] for i in range(len(results.values()))])
variance_processed_neyman = np.sum(variance_list_processed ** (1/2)) ** 2

print(f'''
    Estimator variance using original Hamiltonian                     : {variance_og_neyman}
    Estimator variance using processed Hamiltonians                   : {variance_processed_neyman}

    Number of shots to chemical accuracy using original Hamiltonian   : {variance_og_neyman / 1e-6}
    Number of shots to chemical accuracy using processed Hamiltonians : {variance_processed_neyman / 1e-6}
''')


    Estimator variance using original Hamiltonian                     : (13270428.900770912+0j)
    Estimator variance using processed Hamiltonians                   : (3826.462885149553+1.629460646572709e-06j)

    Number of shots to chemical accuracy using original Hamiltonian   : (13270428900770.912+0j)
    Number of shots to chemical accuracy using processed Hamiltonians : (3826462885.1495533+1.629460646572709j)



Repeat experiment using only non-trivial UCSFs. To determine non-trivial CSFs, look at angles and only use states with at least one angle.

In [11]:
# print angles
list_list_theta_CSF

[array([-0.08241323, -0.04978729, -0.05531925, -0.04468039, -0.03230758,
        -0.01797492, -0.02766998, -0.01098643]),
 array([-0.11172474, -0.15785738, -0.01085365]),
 array([-1.26970468,  0.40270925, -0.32548656]),
 array([0.79054784, 0.03805055, 0.27020658]),
 array([ 0.07040757,  0.00569626, -0.15431912]),
 [],
 [],
 [],
 [],
 [],
 []]

In [12]:
Nqubits = obt.shape[0]
dim     = 2 ** Nqubits
Norb    = Nqubits // 2
Nstates = len(tz_states)

# generate Hamiltonian and obtain its QWC decomposition, for both expectation values and matrix elements

Hferm          = make_short_H_ferm_op(0, obt, tbt)
Hqub           = jordan_wigner(Hferm)
qubit_constant = Hqub.constant
Hqub          -= qubit_constant
Hqub.compress()

qwc_decomp      = sorted_insertion_decomposition(Hqub, 'qwc')
qwc_decomp_swap = augment_decomp_with_pauli_x_plus_i_pauli_y(qwc_decomp, Nqubits)

# generate states and associated state information

configs      = [somos_to_seniority_config(somo, Norb) for somo in somos_list]
statevectors = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in tz_states]

results = {}

for i in range(Nstates):
    for j in range(Nstates):
        if i >= j and i < 5 and j < 5:

            # load the states 
            bra               = statevectors[i]
            bra_tz            = tz_states[i]
            bra_somos         = somos_list[i]
            bra_config        = configs[i]

            ket               = statevectors[j]
            ket_tz            = tz_states[j]
            ket_somos         = somos_list[j]
            ket_config        = configs[j]

            bra_ket_composite = create_composite_state(bra, ket, Nqubits)

            # process the Hamiltonian for measurements
            Hferm_processed        = process_fermionic_hamiltonian_to_remove_irrelevant_terms(Hferm, Nqubits, bra, ket_tz)
            Hqub_temp              = jordan_wigner(Hferm_processed)
            bra_ket_qubit_constant = Hqub_temp.constant
            Hqub_temp             -= bra_ket_qubit_constant
            Hqub_temp.compress()
            Hqub_processed         = process_qubit_hamiltonian_to_remove_irrelevant_terms(Hqub_temp, Nqubits, bra_config, ket_config)
            Hqub_tapered           = process_qubit_hamiltonian_to_project_onto_symmetric_subspace(Hqub_temp, Nqubits, bra_config, ket_config)

            # obtain Hamiltonian fragments
            qwc_decomp_processed      = sorted_insertion_decomposition(Hqub_processed, 'qwc')
            qwc_decomp_processed_swap = augment_decomp_with_pauli_x_plus_i_pauli_y(qwc_decomp_processed, Nqubits)

            # # do a bunch of correctness checks

            # # first: should get the same matrix element no matter how Hamiltonian is processed
            # matrix_element1 = np.round((bra @ get_sparse_operator(Hferm, Nqubits)                                   @ ket.T)[0,0] , 8)
            # matrix_element2 = np.round((bra @ get_sparse_operator(Hferm_processed, Nqubits)                         @ ket.T)[0,0] , 8)
            # matrix_element3 = np.round((bra @ get_sparse_operator(Hqub + qubit_constant, Nqubits)                   @ ket.T)[0,0] , 8)
            # matrix_element4 = np.round((bra @ get_sparse_operator(Hqub_processed + bra_ket_qubit_constant, Nqubits) @ ket.T)[0,0] , 8)

            # assert matrix_element1 == matrix_element2
            # assert matrix_element1 == matrix_element3
            # assert matrix_element1 == matrix_element4
            # assert matrix_element2 == matrix_element3
            # assert matrix_element2 == matrix_element4
            # assert matrix_element3 == matrix_element4
                
            # # second: should get the same matrix element from the fragments, including with swap test
            # H_from_fragments           = sum(qwc_decomp_processed)
            # H_tensor_1q_from_fragments = sum(qwc_decomp_processed_swap)

            # assert (Hqub_processed - H_from_fragments) == QubitOperator().zero()
            # assert (Hqub_processed * (QubitOperator(f'X{Nqubits}') + 1j * QubitOperator(f'Y{Nqubits}')) 
            #         - H_tensor_1q_from_fragments) == QubitOperator().zero()


            # matrix_element5 = np.round((bra @ get_sparse_operator(H_from_fragments + bra_ket_qubit_constant, Nqubits) @ ket.T)[0,0], 8)
            # if i == j:
            #     cur_op = (
            #         H_tensor_1q_from_fragments 
            #         + bra_ket_qubit_constant
            #     )
            #     matrix_element6 = np.round((bra_ket_composite @ 
            #                                 get_sparse_operator(cur_op, Nqubits + 1) @ 
            #                                 bra_ket_composite.T)[0,0], 8)
            # else:
            #     cur_op = (
            #         H_tensor_1q_from_fragments 
            #         + bra_ket_qubit_constant * QubitOperator(f'X{Nqubits}') 
            #         + 1j * bra_ket_qubit_constant * QubitOperator(f'Y{Nqubits}')
            #     )
            #     matrix_element6 = np.round((bra_ket_composite @ 
            #                                 get_sparse_operator(cur_op, Nqubits + 1) @ 
            #                                 bra_ket_composite.T)[0,0], 8)

            # assert matrix_element1 == matrix_element5
            # assert matrix_element1 == matrix_element6
            # assert matrix_element2 == matrix_element5
            # assert matrix_element2 == matrix_element6
            # assert matrix_element3 == matrix_element5
            # assert matrix_element3 == matrix_element6
            # assert matrix_element4 == matrix_element5
            # assert matrix_element4 == matrix_element6
            # assert matrix_element5 == matrix_element6

            # now calculate the variance of the QWC decomposition for both the unprocessed Hamiltonian and the processed Hamiltonian

            if i == j:
                var_og        = variance_of_decomp(qwc_decomp, ket, Nqubits, general=True)
                var_processed = variance_of_decomp(qwc_decomp_processed, ket, Nqubits, general=True)
            
            else:
                var_og        = variance_of_decomp(qwc_decomp_swap, bra_ket_composite, Nqubits + 1, general=True)
                var_processed = variance_of_decomp(qwc_decomp_processed_swap, bra_ket_composite, Nqubits + 1, general=True)

            results[(i,j)] = (bra_config, ket_config, var_processed, var_og)
            print("{:<2} {:<4} {:<10} {:<15}".format(i, j, np.real(np.round(var_processed, 4)), np.real(np.round(var_og, 4))))


0  0    0.415      36.6772        
1  0    0.9939     4416.2446      
1  1    0.2163     35.3914        
2  0    1.5538     4369.4532      
2  1    1.5195     4086.9281      
2  2    0.2893     40.3724        
3  0    1.6681     4544.7063      
3  1    0.8124     4271.1085      
3  2    0.9704     4226.6055      
3  3    0.8394     44.3712        
4  0    0.5015     4617.7143      
4  1    3.0983     4349.5317      
4  2    0.9805     4304.636       
4  3    0.7355     4488.4388      
4  4    0.4071     39.2445        


In [13]:
variance_list_og          = np.array([list(results.values())[i][3] for i in range(len(results.values()))])
variance_og_neyman        = np.sum(variance_list_og ** (1/2)) ** 2

variance_list_processed   = np.array([list(results.values())[i][2] for i in range(len(results.values()))])
variance_processed_neyman = np.sum(variance_list_processed ** (1/2)) ** 2

print(f'''
    Estimator variance using original Hamiltonian                     : {variance_og_neyman}
    Estimator variance using processed Hamiltonians                   : {variance_processed_neyman}

    Number of shots to chemical accuracy using original Hamiltonian   : {variance_og_neyman / 1e-6}
    Number of shots to chemical accuracy using processed Hamiltonians : {variance_processed_neyman / 1e-6}
''')


    Estimator variance using original Hamiltonian                     : (478949.0173995584+0j)
    Estimator variance using processed Hamiltonians                   : (200.8405894327421+0j)

    Number of shots to chemical accuracy using original Hamiltonian   : (478949017399.5584+0j)
    Number of shots to chemical accuracy using processed Hamiltonians : (200840589.4327421+0j)

